In [ ]:
import spacy
from spacy.pipeline import EntityRuler
from spacy.tokens import Doc
import re
import json

In [400]:
with open('..\\list_files\\genus_names.json', 'r', encoding='utf-8') as f:
    genera = json.load(f)
    
with open('..\\list_files\\species_names.json', 'r', encoding='utf-8') as f:
    species = json.load(f)
    
with open('..\\list_files\\compounds_list.json', 'r', encoding='utf-8') as f:
    compounds = json.load(f)
    
with open('..\\list_files\\fatty_acid_list.json', 'r', encoding='utf-8') as f:
    fatty_acids = json.load(f)
    
with open('..\\list_files\\utilizations_list.json', 'r', encoding='utf-8') as f:
    utilizations = json.load(f)

compounds_patterns = []
for comp in compounds:
    compounds_patterns.append({'label': 'COMPOUND', 'pattern': comp})
    
fatty_acids_patterns = []
for fat in fatty_acids:
    fatty_acids_patterns.append({'label': 'MAJOR_FATTY_ACIDS', 'pattern': fat})
    
utilizations_patterns = []
for utilize in utilizations:
    utilizations_patterns.append({'label': 'KIND_OF_UTILIZATION_TESTED', 'pattern': utilize})

with open('..\\list_files\\genus_endings.json', 'r', encoding='utf-8') as f:
    genera_endings = json.load(f)

with open('..\\list_files\\species_5_endings.json', 'r', encoding='utf-8') as f:
    species_endings = json.load(f)
    
with open('..\\list_files\\species_5_beginnings.json', 'r', encoding='utf-8') as f:
    species_beginnings = json.load(f)
    
#with open('list_files\\microbe_patterns.json', 'r', encoding='utf-8') as f:
#    microbes_patterns = json.load(f)
    
compounds = list(set(compounds) - set(fatty_acids))

In [ ]:
model = spacy.load('en_core_web_sm')
ruler = model.add_pipe('entity_ruler', before='ner')

genera = ['Dekkera', 'Pseudomonas', 'Microcystis', 'Lactobacillus']

genera_endings = ['bacillus', 'myces', 'ella', 'bacter', 'ia', 'cola', 'ina', 'aea', 'bium', 'fermentans', 'monas', 'marina', 'olus', 'sinus', 'habitans', 'ilus', 'baculum',
                  'thrix', 'ides', 'illium', 'ora', 'cocus']

regular_words = ['and', 'extra', 'kinda', 'a', 'regular', 'sentence', 'this', 'is', 'the', 'what', 'something', 'about']

species_endings = ['arum', 'ium', 'ina', 'ans', 'isiae']
species_beginnings = ['plantar', 'zem', 'therm', 'cerevi']

species_regex = f'(?=({'|'.join(species_beginnings)})[a-z]*)(?=[a-z]*({'|'.join(species_endings)}))'

#note to self: spacys tokenizer caused some shit with this :/

'''
microbes_patterns = [
    [{'SHAPE': 'X.'}, {'TEXT': {'REGEX': '[a-z]{4,}'}, 'IS_LOWER': True}],
    [{'TEXT': {'REGEX': f'(?=[{'|'.join(species_beginnings)}][a-z]*)(?=[a-z]*[{'|'.join(species_endings)}])'}}],
    [{'TEXT': {'REGEX': '[A-Z]{1}'}, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': '[a-z]{4,}'}, 'IS_LOWER': True}],
    [{'SHAPE': 'Xxxx', 'IS_TITLE': True}, {'TEXT': {'REGEX': '[A-Za-z]+', 'NOT_IN': regular_words}}],
    [{'TEXT': {'IN': genera}}, {'SHAPE': 'xxxx', 'TEXT': {'NOT_IN': regular_words}}],
    [{'TEXT': {'IN': genera}, 'SPACY': True}, {'TEXT': 'sp', 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'NOT_IN': regular_words}, 'IS_SPACE': False}],
    [{'TEXT': {'IN': genera}}, {'SHAPE': 'xxxx', 'TEXT': {'NOT_IN': regular_words}}, {'TEXT': 'subsp', 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': '[A-za-z0-9]'}}],
    [{'TEXT': {'REGEX': f'[A-Z]+[{'|'.join(genera_endings)}]'}}, {'SHAPE': 'xxxx', 'TEXT': {'NOT_IN': regular_words}}],
    [{'TEXT': {'REGEX': '[A-Z][a-z]{0,2}'}, 'SPACY': False}, {'TEXT': '.', 'SPACY': True}, {'TEXT': {'REGEX': f'[a-z]+[{'|'.join(species_endings)}]'}}]
]
'''

microbes_patterns = [
    #[{'TEXT': {'REGEX': '[A-Z]{1}'}, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': '[a-z]{4,}'}, 'IS_LOWER': True}],
    #[{'TEXT': {'REGEX': f'(?=[{'|'.join(species_beginnings)}]{{1}}[a-z]*)(?=[a-z]*[{'|'.join(species_endings)}]{{1}}$)'}}],
    [{'TEXT': {'REGEX': species_regex}}],
    [{'TEXT': {'REGEX': '[A-Z].'}}, {'TEXT': {'REGEX': species_regex}}],
    [{'TEXT': {'REGEX': '[A-Z]'}, 'SPACY': False}, {'TEXT': '.'}, {'TEXT': {'REGEX': species_regex}}]
]

compound_regex_patterns = [
    [{'TEXT': {'REGEX': '[0-9],[0,9]-[A-z]+-[A-z]'}}],
    [{'TEXT': {'REGEX': '[0-9]-[A-z]+'}}, {'TEXT': {'REGEX': '[A-z]+ol'}}]
]

ruler.add_patterns(compounds_patterns)
ruler.add_patterns(fatty_acids_patterns)
ruler.add_patterns(utilizations_patterns)

ruler.add_patterns([{'label': 'MICROBE_NAME', 'pattern': pattern} for pattern in microbes_patterns])

In [402]:
test_microbes = [
    'mixed fermentation using L. thermotolerans and S. cerevisiae, where an increase in glycerol and 2-phenyl ethanol production',
    'Specifically, Lpb. plantarum strains co-metabolized with',
    'Specifically, Lp. plantarum strains co-metabolized with',
    'Specifically, L. plantarum strains',
    'extra words in C. zemplinina',
    'a new microbe called plantarina',
    'will it match Lp . ?',
    'the new L. cockadoodledoo strains',
    'Microcystis sp. K139-l',
    'Pseudomonas bacteria subsp. thesubspeciesname',
    'B. subtilis var. natto'
    'Candidatus Bacterium species',
    'Candidatus bacterium',
    'unidentified marine clone and extra',
    'unidentified Pseudomonas something else',
    'unidentified nitrogen-fixing bacteria',
    'Pseudomonus bacterium subsp. SFKJS',
    'Regular bacteria',
    'What about a normal sentence?',
    'this is kinda a Regular sentence tbh',
    'Lactobacillus regular',
    '. . s d ndsf'
]

for microbe in test_microbes:
    print(microbe)
    doc = model(microbe)
    print([t.text for t in doc])
    #print([(t, t.whitespace_) for t in doc])
    #matches = matcher(doc, as_spans=True)
    print([(t.label_, t) for t in doc.ents])

mixed fermentation using L. thermotolerans and S. cerevisiae, where an increase in glycerol and 2-phenyl ethanol production
['mixed', 'fermentation', 'using', 'L.', 'thermotolerans', 'and', 'S.', 'cerevisiae', ',', 'where', 'an', 'increase', 'in', 'glycerol', 'and', '2', '-', 'phenyl', 'ethanol', 'production']
[('KIND_OF_UTILIZATION_TESTED', fermentation), ('MICROBE_NAME', L. thermotolerans), ('MICROBE_NAME', S. cerevisiae), ('KIND_OF_UTILIZATION_TESTED', increase), ('COMPOUND', glycerol), ('COMPOUND', 2-phenyl ethanol), ('KIND_OF_UTILIZATION_TESTED', production)]
Specifically, Lpb. plantarum strains co-metabolized with
['Specifically', ',', 'Lpb', '.', 'plantarum', 'strains', 'co', '-', 'metabolized', 'with']
[('MICROBE_NAME', Lpb. plantarum)]
Specifically, Lp. plantarum strains co-metabolized with
['Specifically', ',', 'Lp', '.', 'plantarum', 'strains', 'co', '-', 'metabolized', 'with']
[('MICROBE_NAME', Lp. plantarum)]
Specifically, L. plantarum strains
['Specifically', ',', 'L.', '